# Foundations · Identity and access

API keys are the most common way cloud credentials leak: pasted in a notebook, committed to Git, shared in a chat. This course uses **no keys**. Instead:

| Term | Meaning | In our setup |
|---|---|---|
| Microsoft Entra ID | Azure's identity service | Your university account |
| Principal | Someone or something that can sign in | You; the app's **managed identity** `id-…` |
| Token | Short-lived proof of identity (about an hour), for one audience | What `DefaultAzureCredential` fetches |
| Audience (scope) | The service a token is for | `https://ai.azure.com` for Foundry |
| Role | A named set of permissions | **Foundry User** (call models), **AcrPull** (pull images) |
| Role assignment | Principal + role + scope (which resource) | Created by `infra/identity.tf` |

With keys turned off on Foundry (`local_auth_enabled = false`), a request needs a valid token **and** a role assignment. Nothing to leak, nothing to rotate.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()
out = course.outputs()

## 1 · Look inside a token

A token is a signed JSON document. The middle part is readable: it says who you are (`oid`), who the token is for (`aud`) and when it expires (`exp`). We print the claims, never the token.

In [ ]:
import base64
import json
from datetime import datetime

from azure.identity import DefaultAzureCredential

token = DefaultAzureCredential().get_token("https://ai.azure.com/.default")
payload = token.token.split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
for key in ["aud", "iss", "oid", "upn", "name", "exp"]:
    value = claims.get(key)
    if key == "exp":
        value = f"{value}  ({datetime.fromtimestamp(value):%H:%M})"
    print(f"{key:5s} {value}")

## 2 · Where `DefaultAzureCredential` gets it

It tries a list of sources in order and uses the first that works:

1. Environment variables (service principals in CI).
2. Workload identity (Kubernetes).
3. **Managed identity**: in Azure. Our Container App sets `AZURE_CLIENT_ID` so it picks the user-assigned identity `id-…`.
4. **Azure CLI**: on your laptop, your `az login`.

Same code, two places, no secrets.

## 3 · Who can do what on your Foundry resource

In [ ]:
foundry_id = course.run(
    ["az", "cognitiveservices", "account", "show", "-n", out["foundry_account_name"], "-g", out["resource_group_name"], "--query", "id", "-o", "tsv"],
    quiet=True,
).strip()
course.run(["az", "role", "assignment", "list", "--scope", foundry_id, "--query", "[].{who:principalName, type:principalType, role:roleDefinitionName}", "-o", "table"])

You should see two assignments of **Foundry User** (shown as "Azure AI User" while Microsoft renames it): one for you, one for the app's managed identity. That is least privilege: enough to call models and run agents, not enough to delete the resource or change who has access.

## 4 · Without a token, nothing works

In [ ]:
import httpx2

r = httpx2.post(f"{out['foundry_openai_endpoint']}responses", json={"model": out["chat_deployment"], "input": "hi"})
print(r.status_code, r.text[:200])

## What to remember

- Identities, tokens and roles replace keys. Tokens expire within the hour; roles are revoked in one place.
- `DefaultAzureCredential` makes the same code work on your laptop (your login) and in Azure (managed identity).
- Grant the smallest role that works, on the smallest scope that works.

**Learn more:** [Role-based access control for Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/concepts/rbac-foundry) · [Image pull from ACR with managed identity](https://learn.microsoft.com/en-us/azure/container-apps/managed-identity-image-pull)